# Safety-Incident Anomaly Pattern Detection

**INT396 · SIH26165**  
This notebook is part of a reproducible project comparing Isolation Forest and Local Outlier Factor on OSHA Severe Injury Reports.

**Interpretation rule:** model flags are statistical outliers, not verified anomalies, accident predictions, or proof of unsafe conduct.

Select the project's Python environment as the notebook kernel. The raw dataset must be available at `data/raw/January2015toNovember2025.csv`.

# Anomaly Detection Experiments

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
from safety_anomaly.data.io import load_incidents
from safety_anomaly.data.clean import prepare_records
from safety_anomaly.features.build import build_features
from safety_anomaly.models.experiments import run_experiment_grid
DATA_PATH=ROOT/'data/raw/January2015toNovember2025.csv'
df=prepare_records(load_incidents(DATA_PATH))
n=min(5000,len(df))
sample=df.sample(n=n,random_state=42).sort_index().reset_index(drop=True)
features=build_features(sample, representation='structured_text', random_state=42)
X=features.X
print('Sample size:',len(sample),'feature matrix:',X.shape)

## 1. Quick parameter grid
The notebook uses a smaller grid first. Increase the grid only after measuring execution time and memory. LOF is bounded to the sample size configured for the run.

In [ ]:
summary, predictions=run_experiment_grid(X, random_state=42, quick=True, lof_max_rows=10000)
display(summary.sort_values(['model','anomaly_percent']))

## 2. Compare reference configurations

In [ ]:
if_key=next(k for k in predictions if k.startswith('iforest_n100_c0.03'))
lof_key=next(k for k in predictions if k.startswith('lof_k15_c0.03'))
if_result=predictions[if_key]
lof_result=predictions[lof_key]
print('Isolation Forest:', if_key, if_result['result'].parameters)
print('LOF:', lof_key, lof_result['result'].parameters)
print('IF flagged:', int((if_result['labels']==-1).sum()))
print('LOF flagged:', int((lof_result['labels']==-1).sum()))

## 3. Export experiment summary

In [ ]:
out=ROOT/'artifacts/public'
out.mkdir(parents=True,exist_ok=True)
summary.to_csv(out/'experiment_summary.csv',index=False)
print('Saved:',out/'experiment_summary.csv')

## Interpretation
Contamination sets the expected share of flagged observations in these estimators. It is not an estimate of the true anomaly prevalence. Compare detection counts and scores as exploratory model behavior, not accuracy.